##导入相关包

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

# 中研院CKIP要開的東西

In [ ]:
!pip install ckiptagger
!pip install ckipnlp
!pip install ckip_classic

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument

pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})# 不會用到

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER

In [ ]:
ws = WS("/content/drive/MyDrive/CKIP/data")
pos = POS("/content/drive/MyDrive/CKIP/data")
ner = NER("/content/drive/MyDrive/CKIP/data")

# 中研院CKIP作特徵選取

In [ ]:
def outputToDes(Alist,destination):
  with open(destination, 'w', encoding='utf-8') as f:
    json.dump(Alist, f, ensure_ascii=False, indent=4)
  return

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
import re
import json

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞後的資料/nchu+nuk+nkust+開放政府資料.json", "r") as f:
    data = json.load(f)
# 只有篩出來的詞(依據詞性)
jsonDoc1 = []
# 只有篩出來的詞
destination1="/content/drive/MyDrive/collab-topic/CKIP後/(POS)特徵選取詞性/nchu+nuk+nkust+開放政府資料2.json"
jsonDoc2 = []

selected_words = []
a=0
for item in data:
    if a==2: break
    a+=1
    selected_words.clear()
    title_and_desc = item.get("TitleAndDESC", [])
    # 將文本轉換為詞列表
    word_list = title_and_desc.split()
    # print("word_list:\n",word_list)
    # 篩選出名詞和動詞
    pos_list = pos(word_list)
    print("pos_list:\n",pos_list)
    # print(pos_list)
    for i, sent_pos in enumerate(pos_list):
      for word_pos in sent_pos:
          if word_pos in {"Na", "Nb","Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VF", "VE", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}:
              selected_words.append(word_list[i])
              break

    # print("selected_words:\n",selected_words)
    jsonDoc1.append({
        "TitleAndDESC":selected_words.copy(),
        "Topic":item.get("Topic", [])
    })
    # entity_sentence_list = ner(word_list, pos_list)
    # print(entity_sentence_list)

# outputToDes(jsonDoc1,destination1)

# print(selected_words)



## Gensim LDA

### 超參數組合

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

In [ ]:
def outputToTxt(content,d):# 變量內容, 目的地：用於將變量輸出到文字檔裡面觀察
  with open(d, "w") as f:
    f.write(str(content))
  return

In [ ]:
# 用來創建(看)變數-texts、corpus、dictionary
import json
from gensim import corpora, models

# 讀取 JSON 檔案
with open('/content/drive/MyDrive/collab-topic/CKIP後/(POS)特徵選取詞性/nchu+nuk+nkust+開放政府資料.json', 'r') as f:
    data = json.load(f)

# 將詞彙列表合併為字串
docs = [" ".join(d["TitleAndDESC"]) for d in data]
# print("docs:\n",docs)
outputToTxt(docs,'docs.txt')

# 創建詞袋
texts = [[word for word in doc.split()] for doc in docs]
# print("texts:\n",texts)
dictionary = corpora.Dictionary(texts)
outputToTxt(texts,'texts.txt')
# print("dictionary:\n",dictionary)
outputToTxt(dictionary,'dictionary.txt')
corpus = [dictionary.doc2bow(text) for text in texts]
# print("corpus:\n",corpus)
outputToTxt(corpus,'corpus.txt')

# 創建LDA模型
# lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=10)

dictionary 的樣子如下舉例
{'apple': 0, 'juice': 1, 'orange': 2, 'banana': 3}

词袋（bag of words）表示的函数。它的作用是将文档（一个列表，其中包含一些字符串）转换为词袋（**即一个词汇表中每个单词出现的次数**）。

In [ ]:
# 單進程版
import itertools
import numpy as np
from gensim.models import LdaModel
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel

# 定义超参数空间
param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [100],
        'passes': [100]}

# 枚举所有超参数组合
param_list = list(itertools.product(*param_grid.values()))

# 定义模型评分函数
def evaluate_model(param):
    lda_model = LdaModel(corpus, id2word=dictionary, random_state=42, **dict(zip(param_grid.keys(), param)))
    coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
    return coherence_model.get_coherence()

# 计算所有超参数组合的模型评分
scores = np.array([evaluate_model(param) for param in param_list])

# 找到最优超参数组合
best_param = param_list[scores.argmax()]
print(f'Best param: {dict(zip(param_grid.keys(), best_param))}, score: {scores.max()}')


In [ ]:
print(scores)

#### 訓練模型

In [ ]:
lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=7, iterations=300, passes=300, random_state=42)

zip是Python內建函數，可以將多個可迭代對象合併，返回一個元組的列表，其中每個元組包含各可迭代對象相同位置上的元素。例如，如果有兩個列表a=[1,2,3]和b=['a','b','c']，則zip(a,b)的返回值是[(1,'a'),(2,'b'),(3,'c')]。

## 可視化

In [ ]:
!pip install pyLDAvis

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

# 創建可視化資料
vis_data = gensimvis.prepare(topic_model=lda_model, corpus=corpus, dictionary=dictionary)

# 将数据保存到JSON文件

pyLDAvis.save_json(vis_data, '/content/drive/MyDrive/collab-topic/主題模型實驗結果/0226可視化結果(num_topics=7, iterations=300, passes=300).json')
# 顯示可視化
pyLDAvis.display(vis_data)

# 保存為HTML文件
pyLDAvis.save_html(vis_data, '/content/drive/MyDrive/collab-topic/主題模型實驗結果/0226collab-topic(num_topics=7, iterations=300, passes=300).html')
# 儲存模型
lda_model.save('/content/drive/MyDrive/collab-topic/主題模型實驗結果/0226collab-topic(num_topics=7, iterations=300, passes=300).gensim')

## 主題內容(字)

In [ ]:
def outputToTxt(content,d):# 變量內容, 目的地：用於將變量輸出到文字檔裡面觀察
  with open(d, "w") as f:
    f.write(str(content))
  return

In [ ]:
topicContent=lda_model.print_topics(num_topics=7, num_words=100)# num_topics主題數量, num_words主題內的字詞數量
outputToTxt(topicContent,"/content/drive/MyDrive/collab-topic/主題模型實驗結果/0226topicContent(num_topics=7, iterations=300, passes=300).txt")

# 預測新文本

In [ ]:
!pip install opencc

In [ ]:
import opencc
import re
import jieba
from ckiptagger import data_utils, construct_dictionary, WS, POS
import re

# 載入停用詞
with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", encoding="utf-8") as f:
    stop_words = set(f.read().split())

# 載入詞典
# word_to_weight = {"的": 1, "了": 1, "在": 1, "是": 1, "有": 1, "我": 1, "和": 1, "你": 1, "他": 1, "她": 1, "它": 1}
# dictionary = construct_dictionary(word_to_weight=word_to_weight)
# ws = WS("/content/drive/MyDrive/CKIP/data")
# pos = POS("/content/drive/MyDrive/CKIP/data")

# 清洗新文本用
def preprocess_text(text):
    # 刪除英文
    text = re.sub(r'[a-zA-Z]', '', text)

    # 簡體轉繁體
    # text = opencc.convert(text, config='s2t.json')

    # 斷詞、詞性標示
    sentence_list = ws(
        [text], 
        sentence_segmentation=True, 
        segment_delimiter_set={'。', '，', '！', '？'}
    )[0]
    print("sentence_list:\n",sentence_list)
    pos_list = pos(sentence_list)
    print("pos_list:\n",pos_list)

    # 過濾掉不需要的詞性
    valid_pos = {"Na", "Nb", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VF", "VE", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}
    words = []
    for i, sentence in enumerate(pos_list):
        for p in sentence:
            if p in valid_pos:
                words.append(sentence_list[i])
                break

    # selected_words = [word_list[i] for i, sent_pos in enumerate(pos_list) for word_pos in sent_pos
    #               if word_pos in {"Na", "Nb","Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VF", "VE", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}]


    # 過濾停用詞
    words = [w for w in words if w not in stop_words]

    return words


In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import gensim.downloader as api

# 加載訓練好的主題模型
# model_name = 'lda_model'
# model = api.load(model_name)
model = lda_model
# 創建新文本
new_text = '''藝術的契約｜插畫及COSPLAY 展 你也喜歡二次元的東西嗎 ? 或者只是路過而已；不
妨都來看看這場展覽，有用自身演出二次元角色靈
魂的 COSPLAY，以及在平面上繪製出天馬行空想法
或是熱愛的角色的插畫，每一份作品都是創作者對
於這個文化的熱愛，希望您前往現場看看，並能夠
對我們有更多的認識並協助推廣這份熱情 ! ( 線上展
覽同步展出喔 !)'''
print(new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print(new_doc)
# dictionary = Dictionary.load(train_dictionary)
# dictionary = dictionary

# 將新文本轉換為詞袋表示
new_doc_bow = dictionary.doc2bow(new_doc)

# 預測新文本的主題
doc_topics = model.get_document_topics(new_doc_bow)

# 輸出主題預測結果
print(doc_topics)
